In [17]:
# LOADIN THE DATA
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.impute import SimpleImputer

# Load the data
print("Loading data...")
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

Loading data...
Train shape: (614, 13)
Test shape: (367, 12)


In [18]:
# ============================================================
# STEP 1: Create clean copies
# ============================================================
train_clean = train_df.copy()
test_clean = test_df.copy()


In [19]:
# ============================================================
# STEP 2: Identify columns and handle missing values
# ============================================================
print("\n" + "=" * 60)
print("HANDLING MISSING VALUES - AGGRESSIVE APPROACH")
print("=" * 60)

# Get all numeric columns
numeric_cols = train_clean.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nNumeric columns: {numeric_cols}")

# Get all categorical columns (object type)
categorical_cols = train_clean.select_dtypes(include=['object']).columns.tolist()

# Remove Loan_ID and Loan_Status from categorical columns
if 'Loan_ID' in categorical_cols:
    categorical_cols.remove('Loan_ID')
if 'Loan_Status' in categorical_cols:
    categorical_cols.remove('Loan_Status')

print(f"Categorical columns to encode: {categorical_cols}")




HANDLING MISSING VALUES - AGGRESSIVE APPROACH

Numeric columns: ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History']
Categorical columns to encode: ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area']


C:\Users\muham\AppData\Local\Temp\ipykernel_10976\3060256032.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = train_clean.select_dtypes(include=['object']).columns.tolist()


In [20]:
# ============================================================
# STEP 3: Fill missing values with FORCE method
# ============================================================
print("\n" + "=" * 60)
print("FILLING MISSING VALUES")
print("=" * 60)

# For numeric columns - use median
for col in numeric_cols:
    if train_clean[col].isnull().sum() > 0:
        median_val = train_clean[col].median()
        train_clean[col] = train_clean[col].fillna(median_val)
        # Apply to test too
        if col in test_clean.columns:
            test_clean[col] = test_clean[col].fillna(median_val)
        print(f"  Filled '{col}' with median: {median_val}")

# For categorical columns - use mode (most frequent)
for col in categorical_cols:
    if train_clean[col].isnull().sum() > 0:
        mode_val = train_clean[col].mode()[0]
        train_clean[col] = train_clean[col].fillna(mode_val)
        # Apply to test too
        if col in test_clean.columns:
            test_clean[col] = test_clean[col].fillna(mode_val)
        print(f"  Filled '{col}' with mode: {mode_val}")




FILLING MISSING VALUES
  Filled 'LoanAmount' with median: 128.0
  Filled 'Loan_Amount_Term' with median: 360.0
  Filled 'Credit_History' with median: 1.0
  Filled 'Gender' with mode: Male
  Filled 'Married' with mode: Yes
  Filled 'Dependents' with mode: 0
  Filled 'Self_Employed' with mode: No


In [21]:
# ============================================================
# STEP 4: ENSURE NO NaNs REMAIN - EMERGENCY FIX
# ============================================================
print("\n" + "=" * 60)
print("EMERGENCY CHECK - FORCING ALL NaNs TO BE REMOVED")
print("=" * 60)

# Check training data
train_nans = train_clean.isnull().sum().sum()
print(f"NaNs in training before emergency fix: {train_nans}")

if train_nans > 0:
    print("Applying emergency fix to training data...")
    # Fill ANY remaining NaNs with 0 for numeric, 'Unknown' for categorical
    for col in train_clean.columns:
        if train_clean[col].dtype in ['int64', 'float64']:
            train_clean[col] = train_clean[col].fillna(0)
        else:
            train_clean[col] = train_clean[col].fillna('Unknown')
    print("  Emergency fix applied to training")

# Check test data
test_nans = test_clean.isnull().sum().sum()
print(f"NaNs in testing before emergency fix: {test_nans}")

if test_nans > 0:
    print("Applying emergency fix to test data...")
    for col in test_clean.columns:
        if test_clean[col].dtype in ['int64', 'float64']:
            test_clean[col] = test_clean[col].fillna(0)
        else:
            test_clean[col] = test_clean[col].fillna('Unknown')
    print("  Emergency fix applied to test")

# Final verification
print(f"\n✅ Final NaN count - Training: {train_clean.isnull().sum().sum()}")
print(f"✅ Final NaN count - Testing: {test_clean.isnull().sum().sum()}")

assert train_clean.isnull().sum().sum() == 0, "Still have NaNs!"
assert test_clean.isnull().sum().sum() == 0, "Still have NaNs!"
print("✅ All NaNs have been eliminated!")



EMERGENCY CHECK - FORCING ALL NaNs TO BE REMOVED
NaNs in training before emergency fix: 0
NaNs in testing before emergency fix: 0

✅ Final NaN count - Training: 0
✅ Final NaN count - Testing: 0
✅ All NaNs have been eliminated!


In [22]:
# ============================================================
# STEP 5: Encode categorical variables
# ============================================================
print("\n" + "=" * 60)
print("ENCODING CATEGORICAL VARIABLES")
print("=" * 60)

# Encode categorical features
label_encoders = {}
for col in categorical_cols:
    label_encoders[col] = LabelEncoder()
    # Fit on training data
    label_encoders[col].fit(train_clean[col])
    # Transform both
    train_clean[col] = label_encoders[col].transform(train_clean[col])
    test_clean[col] = label_encoders[col].transform(test_clean[col])
    print(f"  Encoded '{col}'")

# Encode target variable
target_encoder = LabelEncoder()
train_clean['Loan_Status'] = target_encoder.fit_transform(train_clean['Loan_Status'])
print(f"  Encoded 'Loan_Status'")
print(f"  Mapping: Y={target_encoder.transform(['Y'])[0]}, N={target_encoder.transform(['N'])[0]}")




ENCODING CATEGORICAL VARIABLES
  Encoded 'Gender'
  Encoded 'Married'
  Encoded 'Dependents'
  Encoded 'Education'
  Encoded 'Self_Employed'
  Encoded 'Property_Area'
  Encoded 'Loan_Status'
  Mapping: Y=1, N=0


In [23]:
# ============================================================
# STEP 6: Prepare features
# ============================================================
print("\n" + "=" * 60)
print("PREPARING FEATURES")
print("=" * 60)

# Define features (exclude Loan_ID and Loan_Status)
feature_cols = [col for col in train_clean.columns if col not in ['Loan_ID', 'Loan_Status']]
print(f"Features to use: {feature_cols}")

X = train_clean[feature_cols]
y = train_clean['Loan_Status']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nClass distribution:")
print(y.value_counts(normalize=True))




PREPARING FEATURES
Features to use: ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Property_Area']
X shape: (614, 11)
y shape: (614,)

Class distribution:
Loan_Status
1    0.687296
0    0.312704
Name: proportion, dtype: float64


In [24]:
# ============================================================
# STEP 7: Create validation split
# ============================================================
print("\n" + "=" * 60)
print("CREATING VALIDATION SPLIT")
print("=" * 60)

X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X, y, 
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training subset: {X_train_sub.shape}")
print(f"Validation subset: {X_val.shape}")

# FINAL CHECK - Verify NO NaNs in split data
print("\n" + "=" * 60)
print("FINAL VERIFICATION BEFORE TRAINING")
print("=" * 60)

print(f"NaNs in X_train_sub: {X_train_sub.isnull().sum().sum()}")
print(f"NaNs in X_val: {X_val.isnull().sum().sum()}")
print(f"NaNs in y_train_sub: {y_train_sub.isnull().sum()}")
print(f"NaNs in y_val: {y_val.isnull().sum()}")

if X_train_sub.isnull().sum().sum() == 0:
    print("\n✅ DATA IS CLEAN! Proceeding to training...")
else:
    print("\n❌ STILL HAVE NaNs! Applying final cleanup...")
    X_train_sub = X_train_sub.fillna(0)
    X_val = X_val.fillna(0)
    print("   Applied fillna(0) as last resort")



CREATING VALIDATION SPLIT
Training subset: (491, 11)
Validation subset: (123, 11)

FINAL VERIFICATION BEFORE TRAINING
NaNs in X_train_sub: 0
NaNs in X_val: 0
NaNs in y_train_sub: 0
NaNs in y_val: 0

✅ DATA IS CLEAN! Proceeding to training...


In [25]:
# ============================================================
# STEP 8: TRAIN THE MODEL (THIS WILL WORK NOW)
# ============================================================
print("\n" + "=" * 60)
print("STEP 8: TRAINING LOGISTIC REGRESSION MODEL")
print("=" * 60)

# Create the model
model = LogisticRegression(max_iter=1000, random_state=42)

# Train the model - THIS SHOULD WORK NOW
model.fit(X_train_sub, y_train_sub)

print("✅ Model training completed successfully!")
print(f"Number of features: {X_train_sub.shape[1]}")
print(f"Model intercept: {model.intercept_[0]:.4f}")



STEP 8: TRAINING LOGISTIC REGRESSION MODEL
✅ Model training completed successfully!
Number of features: 11
Model intercept: -1.5734


C:\Users\muham\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [26]:
# ============================================================
# STEP 9: EVALUATE ON VALIDATION SET
# ============================================================
print("\n" + "=" * 60)
print("STEP 9: MODEL EVALUATION")
print("=" * 60)

# Make predictions
y_pred_val = model.predict(X_val)

# Calculate accuracy
accuracy = accuracy_score(y_val, y_pred_val)
print(f"Validation Accuracy: {accuracy*100:.2f}%")

# Confusion Matrix
cm = confusion_matrix(y_val, y_pred_val)
print("\nConfusion Matrix:")
print(cm)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_val, y_pred_val, target_names=['Not Approved', 'Approved']))




STEP 9: MODEL EVALUATION
Validation Accuracy: 86.18%

Confusion Matrix:
[[22 16]
 [ 1 84]]

Classification Report:
              precision    recall  f1-score   support

Not Approved       0.96      0.58      0.72        38
    Approved       0.84      0.99      0.91        85

    accuracy                           0.86       123
   macro avg       0.90      0.78      0.81       123
weighted avg       0.88      0.86      0.85       123



In [27]:
# ============================================================
# STEP 10: RETRAIN ON ALL DATA AND PREDICT TEST
# ============================================================
print("\n" + "=" * 60)
print("STEP 10: FINAL PREDICTIONS ON TEST DATA")
print("=" * 60)

# Retrain on all training data
model.fit(X, y)
print("Model retrained on all training data!")

# Make predictions on test data
X_test_final = test_clean[feature_cols]
y_pred_final = model.predict(X_test_final)

# Convert back to original labels
y_pred_labels = target_encoder.inverse_transform(y_pred_final)

print(f"\nPredictions made for {len(y_pred_labels)} test applicants")
print("\nPrediction distribution:")
print(pd.Series(y_pred_labels).value_counts())



STEP 10: FINAL PREDICTIONS ON TEST DATA
Model retrained on all training data!

Predictions made for 367 test applicants

Prediction distribution:
Y    308
N     59
Name: count, dtype: int64


C:\Users\muham\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [28]:
# ============================================================
# STEP 11: CREATE SUBMISSION FILE
# ============================================================
print("\n" + "=" * 60)
print("STEP 11: CREATING SUBMISSION FILE")
print("=" * 60)

submission = pd.DataFrame({
    'Loan_ID': test_clean['Loan_ID'],
    'Loan_Status': y_pred_labels
})

submission.to_csv('loan_prediction_submission.csv', index=False)
print("✅ Submission file created: 'loan_prediction_submission.csv'")
print("\nFirst 10 predictions:")
print(submission.head(10))

print("\n" + "=" * 60)
print("TASK 2 COMPLETED SUCCESSFULLY!")
print("=" * 60)


STEP 11: CREATING SUBMISSION FILE
✅ Submission file created: 'loan_prediction_submission.csv'

First 10 predictions:
    Loan_ID Loan_Status
0  LP001015           Y
1  LP001022           Y
2  LP001031           Y
3  LP001035           Y
4  LP001051           Y
5  LP001054           Y
6  LP001055           Y
7  LP001056           N
8  LP001059           Y
9  LP001067           Y

TASK 2 COMPLETED SUCCESSFULLY!
